# The PyTorch training loop

_PyTorch (a complete course)_

**The hand-written loop at the heart of every PyTorch project.**

Training a network is a simple cycle repeated many times: guess, measure the error, find which way to nudge each weight, nudge. The loop just runs that cycle over your data, again and again.

---

This notebook is a practice scaffold for the **The PyTorch training loop** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)                       # reproducible
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- tiny synthetic 2-class dataset (two Gaussian blobs in 8-D) ---
N, D = 1000, 8
centers = torch.randn(2, D) * 2.0
y = torch.randint(0, 2, (N,))              # class labels 0 / 1
X = centers[y] + torch.randn(N, D)         # features near each class center

# train / validation split
ntr = 800
train_ds = TensorDataset(X[:ntr], y[:ntr])
val_ds   = TensorDataset(X[ntr:], y[ntr:])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)

# --- model, loss, optimizer ---
model = nn.Sequential(
    nn.Linear(D, 32), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(32, 2),                      # raw logits (no softmax!)
).to(device)
criterion = nn.CrossEntropyLoss()          # expects logits + int labels
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

def run_epoch(loader, train):
    model.train() if train else model.eval()   # dropout/batchnorm mode
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)   # move to device
            if train:
                optimizer.zero_grad()               # clear old grads
            pred = model(xb)                        # forward
            loss = criterion(pred, yb)              # measure error
            if train:
                loss.backward()                     # backprop
                optimizer.step()                    # update weights
            total_loss += loss.item() * len(xb)     # .item() -> no graph
            correct += (pred.argmax(1) == yb).sum().item()
            n += len(xb)
    return total_loss / n, correct / n

EPOCHS = 15
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    va_loss, va_acc = run_epoch(val_dl,   train=False)
    print(f"epoch {epoch:2d} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")


## Visualize the data & results

_Across epochs, does training loss keep falling while validation loss eventually turns back up?_

In [ ]:
import numpy as np

rng = np.random.RandomState(1)
d = 40                                 # 40 features...
ntr, nva = 30, 300                     # ...but tiny train set -> easy to overfit
w_true = np.zeros(d); w_true[:3] = [1.6, -1.4, 1.0]   # only 3 real signals

def make(n):
    X = rng.randn(n, d)
    logits = X @ w_true + 0.6 * rng.randn(n)
    return X, (logits > 0).astype(float)

Xtr, ytr = make(ntr)
Xva, yva = make(nva)
Xtr = np.hstack([Xtr, np.ones((ntr, 1))])   # bias column
Xva = np.hstack([Xva, np.ones((nva, 1))])

w = np.zeros(d + 1)
sig = lambda z: 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
def bce(X, y, w):
    p = np.clip(sig(X @ w), 1e-7, 1 - 1e-7)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

lr, epochs = 0.3, 120
train_loss, val_loss = [], []
for e in range(epochs):
    p = sig(Xtr @ w)
    grad = Xtr.T @ (p - ytr) / ntr      # logistic gradient
    w -= lr * grad                      # the "optimizer.step()"
    train_loss.append(bce(Xtr, ytr, w))
    val_loss.append(bce(Xva, yva, w))   # held-out: no weight update

# subsample to <= 60 points for the chart (every 2nd epoch)
es = range(0, epochs, 2)
print("train:", [(e + 1, round(train_loss[e], 4)) for e in es])
print("val  :", [(e + 1, round(val_loss[e], 4)) for e in es])
print("val min at epoch", int(np.argmin(val_loss)) + 1,
      "=", round(min(val_loss), 3))


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Put these five per-batch steps in the correct order: loss.backward(), optimizer.step(), pred = model(x), optimizer.zero_grad(), loss = criterion(pred, y).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Start by clearing old gradients. — _Gradients accumulate; if you do not zero them, this batch's update mixes in the last batch's slopes._
- Forward pass to get the prediction, then compute the loss. — _The forward pass builds the autograd graph that backward() will later walk._
- Call backward(), then step(). — _backward() fills each weight's .grad; step() uses those grads to update the weights._

**Answer:** zero_grad() &rarr; pred = model(x) &rarr; loss = criterion(pred, y) &rarr; loss.backward() &rarr; optimizer.step().

</details>

**Problem 2.** Your validation loss looks fine on screen but the script slowly runs out of memory over many epochs. You accumulate the loss with total += loss. What is wrong and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that loss is a tensor still attached to its computation graph. — _Summing it keeps a reference to every batch's graph, so none can be freed._
- Replace it with the scalar value. — _loss.item() returns a plain Python float with no graph attached._

**Answer:** Accumulate loss.item() instead of loss. Also wrap validation in with torch.no_grad(): so no graph is built at all.

</details>

**Problem 3.** You forgot to call model.eval() before validating a network that uses dropout. Why are your validation numbers unreliable?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what dropout does in training mode. — _In train mode it randomly zeroes some activations each forward pass, so the same input gives different outputs._
- Realize validation should be deterministic. — _model.eval() turns dropout off (and switches batch norm to running statistics), giving stable, fair numbers._

**Answer:** Without model.eval(), dropout stays active during validation, so predictions are noisy and the loss/accuracy are not trustworthy. Call model.eval() first, then model.train() again before the next training epoch.

</details>